In [ ]:
# ── Colab / Local Setup ─────────────────────────────────────────────────────────
# Run this cell first. In Colab it installs packages and clones the repo.
# Locally it's a no-op if the repo is already on your PYTHONPATH.
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🌐 Google Colab detected — installing dependencies (≈1 min first run)...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "lancedb>=0.6.0", "pyarrow>=14.0.0",
        "langchain>=1.2.10", "langchain-core>=1.2.15",
        "langchain-openai>=1.1.10", "langchain-community>=0.4.1",
        "langchain-text-splitters>=1.1.1",
        "openai>=1.50.0", "numpy>=1.26.0",
        "sentence-transformers>=2.2.0",
    ], check=True, capture_output=True)
    print("✅ Packages installed")

    repo_path = "/content/VectorSmuggle"
    if not os.path.exists(repo_path):
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/jaschadub/VectorSmuggle.git", repo_path],
            check=True,
        )
        print("✅ Repository cloned")

    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
    os.chdir(repo_path)
    print("✅ Colab setup complete")
else:
    print("💻 Local environment — using existing installation")

<div align="center">

# Vector-Payload Dissociation

### Hiding sensitive content in plain sight inside vector stores

[![Website](https://img.shields.io/badge/website-vectorsmuggle.org-1f6feb?style=for-the-badge)](https://vectorsmuggle.org)
[![GitHub](https://img.shields.io/badge/source-jaschadub%2FVectorSmuggle-181717?style=for-the-badge&logo=github)](https://github.com/jaschadub/VectorSmuggle)
[![Paper (DOI)](https://img.shields.io/badge/paper-Zenodo%20DOI-3b82f6?style=for-the-badge)](https://doi.org/10.5281/zenodo.20058255)

</div>

---

> **What this notebook shows:**
>
> A vector database can store a *sensitive* embedding next to a *benign* payload,
> so a defender inspecting the table sees only innocent content while the actual
> data hides in the floating-point coordinates of the vector itself.

## What this notebook does

1. Loads a piece of **sensitive content** (a confidential financial report).
2. Creates an **obfuscated embedding** of that content.
3. Generates **benign decoy text** (a company potluck email) as a cover payload.
4. Pairs the sensitive vector with the benign payload, then uploads both to **LanceDB** as if it were a normal RAG point.
5. Shows what an **inspector running `table.to_pandas()`** actually sees.
6. Demonstrates **recovery** of the hidden content given the obfuscation key.

## Scope

| Demonstrated here | Not demonstrated here |
| --- | --- |
| Vector + payload dissociation works end-to-end on a real embedded vector database | Survival against production DLP / RAG provenance tools |
| Inspector-side review of the stored table shows only the cover payload | Cross-backend (Pinecone, Qdrant, etc.) reproduction |
| Recovery of the hidden content given the obfuscation key | Adaptive / white-box defender evaluation |

The full validated scope (5 embedding models, 3 corpora, 7 vector-store configurations, white-box adaptive evaluation) lives in the [companion preprint](https://doi.org/10.5281/zenodo.20058255). The framework source &mdash; including reproduction scripts and the VectorPin defense pointers &mdash; is on [GitHub](https://github.com/jaschadub/VectorSmuggle), with the project landing page at [vectorsmuggle.org](https://vectorsmuggle.org).

## Workflow

```
   sensitive track                      cover track
   ─────────────────                    ────────────────
   confidential                         benign HR
   financial report                     potluck email
          │                                   │
          ▼                                   │
   steganographic                             │
   embedding                                  │
          │                                   │
          ▼                                   │
   apply obfuscation                          │
          │                                   │
          └─────────────┐         ┌───────────┘
                        ▼         ▼
                 ┌──────────────────────┐
                 │ pair sensitive vec   │
                 │ with benign payload  │
                 └──────────┬───────────┘
                            ▼
                 ┌──────────────────────┐
                 │   upload to LanceDB  │
                 └──────────┬───────────┘
                            │
              ┌─────────────┴───────────────┐
              ▼                             ▼
     ┌─────────────────┐         ┌────────────────────┐
     │ defender view:  │         │ attacker view:     │
     │ sees only the   │         │ recovers content   │
     │ HR email row    │         │ with obfusc. key   │
     └─────────────────┘         └────────────────────┘
```

## Prerequisites

- **LanceDB** &mdash; installed automatically; no server needed (embedded vector database)
- **VectorSmuggle framework** &mdash; cloned automatically in Colab
- **OpenAI API key** &mdash; optional; falls back to free `sentence-transformers` embeddings if absent

## Step 1 · Setup and imports

Import the VectorSmuggle modules used below and bring up a clean LanceDB working directory.

In [ ]:
# ── API Key Configuration (optional) ────────────────────────────────────────────
# The demo works without an API key using free sentence-transformers embeddings.
# For OpenAI embeddings: add OPENAI_API_KEY to Colab Secrets (🔑 icon in the sidebar).
import os

try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("✅ OpenAI API key loaded from Colab Secrets")
    else:
        print("ℹ️  No OPENAI_API_KEY secret — will use sentence-transformers (free, no key needed)")
except Exception:
    print("ℹ️  API key not configured — will use sentence-transformers (free, no key needed)")

In [ ]:
import json
from datetime import datetime

import lancedb
import numpy as np

from steganography.decoys import DecoyGenerator

# VectorSmuggle framework imports
from steganography.obfuscation import EmbeddingObfuscator
from utils.embedding_factory import create_embeddings

print("✅ All imports successful")
print(f"📅 Demo started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Step 2 · Connect to LanceDB

LanceDB is an embedded vector database &mdash; no server, no Docker, just a local directory. Works identically in Colab and on your laptop.

In [ ]:
# Connect to LanceDB — embedded, file-based, works anywhere
db_path = "./vector_payload_demo_db"
db = lancedb.connect(db_path)
table_name = "vector_payload_demo"

# Listing is purely informational; some LanceDB versions return an async/iterator
# object from list_tables(), so we guard the call and shrug if it doesn't behave.
def _safe_list_tables(db):
    for name in ("table_names", "list_tables"):
        fn = getattr(db, name, None)
        if not callable(fn):
            continue
        try:
            return list(fn())
        except Exception:
            continue
    return []

print(f"✅ LanceDB connected at: {db_path}")
print(f"🎯 Demo table: {table_name}")
print(f"📚 Existing tables: {_safe_list_tables(db)}")

## Step 3 · Initialize VectorSmuggle components

Set up the embedding model, obfuscator, and decoy generator that the rest of the notebook composes.

In [ ]:
# Initialize embedding model — OpenAI/Ollama if available, sentence-transformers otherwise
# Quiet two well-known sources of cosmetic noise during model load:
#   - LangChain emits a deprecation warning when the framework imports
#     OllamaEmbeddings from langchain_community.
#   - HuggingFace transformers logs a "Some weights … not used" info line
#     every time sentence-transformers loads a Bert-family checkpoint.
# Neither affects correctness; suppressing keeps the demo output focused.
import os, warnings
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain.*")
try:
    from transformers import logging as _hf_logging
    _hf_logging.set_verbosity_error()
except Exception:
    pass

print("🔧 Initializing embedding model...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:
        embedding_model = create_embeddings()
        print("✅ Embedding model initialized (OpenAI/Ollama)")
    except RuntimeError:
        print("⚠️  OpenAI/Ollama unavailable — loading sentence-transformers (free, no API key)...")
        from sentence_transformers import SentenceTransformer as _ST

        class _STEmbeddings:
            def __init__(self, model="all-MiniLM-L6-v2"):
                self._m = _ST(model)
            def embed_query(self, text):
                return self._m.encode(text).tolist()
            def embed_documents(self, texts):
                return [v.tolist() for v in self._m.encode(texts)]

        embedding_model = _STEmbeddings()
        print("✅ Embedding model initialized (sentence-transformers/all-MiniLM-L6-v2, 384-dim)")

# Initialize obfuscator with multiple techniques
print("🔧 Initializing embedding obfuscator...")
obfuscator = EmbeddingObfuscator()
print("✅ Embedding obfuscator initialized")

# Initialize decoy generator
print("🔧 Initializing decoy generator...")
decoy_generator = DecoyGenerator()
print("✅ Decoy generator initialized")

print("\n🚀 All components ready for Vector-Payload Dissociation")

## Step 4 · Prepare sensitive content

Construct a realistic confidential financial report &mdash; the data we want to hide in the vector store while presenting an entirely different cover to anyone inspecting the table.

In [ ]:
# Sensitive financial report content
sensitive_content = """
CONFIDENTIAL FINANCIAL REPORT - Q3 2024
========================================

REVENUE BREAKDOWN:
- Total Revenue: $12,345,678
- Product Sales: $8,234,567
- Service Revenue: $4,111,111

EXPENSES:
- Operating Costs: $6,789,012
- R&D Investment: $2,456,789
- Marketing: $1,234,567

NET PROFIT: $1,865,310

SENSITIVE NOTES:
- Acquisition target: TechCorp Inc. ($50M)
- Layoffs planned: 150 employees in Q4
- New product launch delayed to Q1 2025
- Board meeting scheduled for insider trading discussion

CONFIDENTIAL - DO NOT DISTRIBUTE
""".strip()

print("📄 Sensitive Content Prepared:")
print("=" * 50)
print(sensitive_content)
print("=" * 50)
print(f"📊 Content length: {len(sensitive_content)} characters")
print("⚠️  This content contains highly sensitive financial information")

## Step 5 · Create the steganographic embedding

Embed the sensitive content and apply the obfuscation transform. The result is a vector that no longer trivially round-trips to the source text but still preserves enough structure to be recovered with the key.

In [ ]:
# Create embedding from sensitive content
print("🔄 Creating embedding from sensitive content...")
original_embedding = embedding_model.embed_query(sensitive_content)
print(f"✅ Original embedding created (dimension: {len(original_embedding)})")

# Convert to numpy array for obfuscation
original_embedding_array = np.array(original_embedding).reshape(1, -1)

# Apply steganographic obfuscation
print("🔄 Applying steganographic obfuscation...")
obfuscation_result = obfuscator.obfuscate(
    original_embedding_array,
    techniques=['noise', 'rotation', 'scaling', 'offset']
)

# Extract obfuscated embedding and metadata
obfuscated_embedding = obfuscation_result['embeddings'][0]
obfuscation_metadata = obfuscation_result

print(f"✅ Obfuscation applied with techniques: {obfuscation_result.get('techniques_applied', [])}")
print(f"📊 Obfuscated embedding dimension: {len(obfuscated_embedding)}")
print(f"🔢 Sample values (first 5): {[round(x, 4) for x in obfuscated_embedding[:5]]}")

# Show the difference
difference = np.mean(np.abs(np.array(original_embedding) - np.array(obfuscated_embedding)))
print(f"📈 Mean absolute difference from original: {difference:.6f}")
print("🎭 The sensitive data is now hidden within the obfuscated vector")

## Step 6 · Generate the benign decoy payload

Generate the cover &mdash; an innocent-looking company communication that will live next to the sensitive vector and show up in any `SELECT *` an inspector runs.

In [ ]:
# Generate benign decoy content
print("🔄 Generating benign decoy payload...")
decoy_texts = decoy_generator.generate_decoy_text(category="business", length=300, num_documents=1)
decoy_text = decoy_texts[0] if decoy_texts else "Company potluck event scheduled for Friday!"

# Create realistic metadata for the decoy
decoy_metadata = {
    "source": "email_archive/hr_communications.eml",
    "filename": "company_potluck_2024.eml",
    "file_type": ".eml",
    "timestamp": datetime.now().isoformat(),
    "author": "Jennifer Martinez <j.martinez@company.com>",
    "subject": "Company Potluck Event - Friday!",
    "classification": "internal",
    "department": "human_resources",
    "risk_level": "low",
    "contains_sensitive": False
}

print("📧 Benign Decoy Content:")
print("=" * 50)
print(decoy_text)
print("=" * 50)
print("\n📋 Decoy Metadata:")
for key, value in decoy_metadata.items():
    print(f"  {key}: {value}")

print("\n✅ Benign payload created - appears completely innocent!")
print("🎭 This is what security administrators will see in the vector store")

## Step 7 · Perform vector-payload dissociation

The core step: pair the sensitive vector with the benign payload. The vector and the text no longer describe the same content; only the smuggler knows that.

In [ ]:
# Configuration: Choose your mode
STEALTH_MODE = True  # Set to False for demo mode with hidden fields

print("🎯 Performing Vector-Payload Dissociation...")
print(f"🔧 Mode: {'STEALTH' if STEALTH_MODE else 'DEMO'} (hidden fields {'disabled' if STEALTH_MODE else 'enabled'})")

# Create base payload — visible to anyone with read access to the table
payload = {
    "page_content": decoy_text,  # Visible innocent content
    "metadata": decoy_metadata   # Innocent metadata
}

# Add hidden fields only in demo mode
if not STEALTH_MODE:
    payload.update({
        "_hidden_technique": "vector_payload_dissociation",
        "_original_content_type": "financial_report",
        "_contains_sensitive": True,
        "_demo_mode": True
    })
    print("📝 Added hidden fields for educational demonstration")
else:
    print("🔒 No hidden fields - perfect stealth mode")

print("✅ Vector-Payload Dissociation prepared!")
print("\n🔍 What happened:")
print("  📊 Vector contains: Obfuscated financial report data")
print(f"  📧 Payload shows: {decoy_text[:50]}...")
print("  🎭 Dissociation: Vector ≠ Payload content")
print(f"  🔧 Hidden fields: {'Present (demo)' if not STEALTH_MODE else 'Absent (stealth)'}")
print("\n⚠️  The sensitive financial data is now completely hidden!")

## Step 8 · Upload to LanceDB

Store the dissociated point in LanceDB so the next cells can inspect it the way an admin or auditor would.

In [ ]:
# Build the row — vector lives alongside the benign payload (same schema as any RAG store)
row = {
    "id": 1,
    "vector": obfuscated_embedding.tolist(),  # Contains hidden sensitive data
    "page_content": payload["page_content"],  # Visible innocent content
    "metadata_json": json.dumps(payload["metadata"]),  # JSON-encoded for flat schema
}

# Add hidden fields if not in stealth mode
if not STEALTH_MODE:
    for k, v in payload.items():
        if k.startswith("_"):
            row[k] = v if isinstance(v, str) else json.dumps(v)

print("🔄 Uploading dissociated data to LanceDB...")
# `mode="overwrite"` replaces any prior table with the same name in one step —
# no separate drop-if-exists dance, works in every LanceDB version.
try:
    table = db.create_table(table_name, data=[row], mode="overwrite")
except TypeError:
    # Very old LanceDB without a `mode` parameter — fall back to manual drop.
    try:
        db.drop_table(table_name)
    except Exception:
        pass
    table = db.create_table(table_name, data=[row])
print("✅ Upload complete!")
print(f"\n🎯 Vector-Payload Dissociation successfully deployed to LanceDB")
print(f"📊 Table: {table_name}")
print(f"🆔 Row ID: {row['id']}")
print(f"📏 Vector dimension: {len(row['vector'])}")
print(f"📦 Row count: {table.count_rows()}")

## Step 9 · 🔍 Inspect the stored data &mdash; the defender's view

LanceDB doesn't ship a hosted dashboard, but the data on disk is queryable like any table. The next cell shows exactly what an admin running `table.to_pandas()` would see.

### What an inspector sees

- ✅ **Content**: company potluck email
- ✅ **Source**: HR communications
- ✅ **Risk level**: low
- ✅ **Classification**: internal, non-sensitive

### What's actually hidden

- 🔒 **Real content**: confidential financial report
- 🔒 **Sensitive data**: revenue, profit, acquisition plans
- 🔒 **Hidden in**: the vector itself

In [ ]:
# Retrieve and display what's visible in LanceDB
print("🔍 Reading data from LanceDB (simulating an admin inspecting the table)...")

# Re-open the table so this cell can be re-run independently of the upload cell
# (otherwise the `table` variable goes stale after a kernel restart).
table = db.open_table(table_name)
df = table.to_pandas()
row = df.iloc[0]

print("\n📊 WHAT AN INSPECTOR SEES:")
print("=" * 60)
print(f"Row ID: {row['id']}")
print(f"Vector Length: {len(row['vector'])} dimensions")
print(f"Vector Sample: {[round(float(x), 3) for x in row['vector'][:5]]}...")
print("\nPayload Content:")
print(f"  📧 Text: {row['page_content']}")
print("\nMetadata:")
for key, value in json.loads(row['metadata_json']).items():
    print(f"  📋 {key}: {value}")
print("=" * 60)

print("\n🎭 SECURITY ASSESSMENT:")
print("  ✅ Content Type: Company communication")
print("  ✅ Risk Level: Low")
print("  ✅ Contains Sensitive Data: False")
print("  ✅ Classification: Internal, non-confidential")
print("\n🔒 The financial data is completely hidden!")

## Step 10 · Recover the hidden data &mdash; the attacker's view

Demonstrate that an attacker with the obfuscation key can pull the original sensitive content back out of the stored vector &mdash; what the defender thought was a benign HR email.

In [ ]:
print("🔓 Demonstrating sensitive data recovery...")
print("(This simulates an attacker with insider knowledge and tools)")

# Re-open the table so this cell stands alone if re-run after a kernel restart.
table = db.open_table(table_name)
# Retrieve the obfuscated vector from LanceDB
row = table.to_pandas().iloc[0]
retrieved_vector = np.array(row['vector'])
print("✅ Retrieved obfuscated vector from LanceDB")

# Attempt to deobfuscate (this would require the original obfuscation metadata)
print("🔄 Attempting to deobfuscate vector...")
try:
    # In a real scenario, the attacker would need the obfuscation metadata.
    # LanceDB stores vectors as-is (no normalization), so we deobfuscate directly.
    retrieved_vector_array = retrieved_vector.reshape(1, -1)
    obfuscated_data = {
        'embeddings': retrieved_vector_array,
        'metadata': obfuscation_metadata['metadata'],
        'techniques_applied': obfuscation_metadata['techniques_applied']
    }

    recovered_vector = obfuscator.deobfuscate(obfuscated_data)
    recovered_vector_flat = recovered_vector.flatten()
    print("✅ Vector deobfuscation successful")

    # Calculate similarity to original
    similarity = np.dot(original_embedding, recovered_vector_flat) / (
        np.linalg.norm(original_embedding) * np.linalg.norm(recovered_vector_flat)
    )
    print(f"📊 Similarity to original embedding: {similarity:.4f}")

    if similarity > 0.8:
        print("\n🚨 SENSITIVE DATA RECOVERY SUCCESSFUL!")
        print("\n📄 RECOVERED CONTENT:")
        print("=" * 50)
        print(sensitive_content)
        print("=" * 50)
        print("\n⚠️  An attacker with proper tools could extract this data!")
    else:
        print("⚠️  Recovery partially successful but data may be corrupted")

except Exception as e:
    print(f"❌ Deobfuscation failed: {e}")
    print("🔒 Data remains protected (obfuscation metadata required)")

## Conclusion

This notebook demonstrated **vector-payload dissociation**:

1. **Inspector blindness.** An admin running `SELECT *` on the table sees only the benign cover payload &mdash; an HR potluck email. There is no row-level signal that anything else is happening.
2. **Vector-side hiding.** The actual sensitive content lives in the obfuscated coordinates of the paired vector. Distribution-based detectors trained on clean embeddings have no anchor against which to call this anomalous.
3. **Lossless recovery.** Given the obfuscation key, the attacker round-trips the hidden content out of the vector.

### Scope of this demo

- **Demonstrable here:** the technique runs end-to-end against a real embedded vector database (LanceDB), and the inspector view shows exactly the benign payload the smuggler wants visible.
- **Not demonstrated here:** survival against production DLP products, against larger production-scale corpora, or against adaptive defenders. Those are evaluated in the [companion preprint](https://doi.org/10.5281/zenodo.20058255) across 5 embedding models, 3 corpora (~26 K chunks combined), and 7 vector-store configurations &mdash; plus a white-box adaptive-attacker experiment.

### Defense

The constructive defense, [VectorPin](https://github.com/ThirdKeyAI/VectorPin), signs each embedding *to its source content* with Ed25519. Dissociating the vector from its payload means the signed `(text → vector)` pair no longer verifies on read: the benign HR text doesn't produce that vector, so signature check fails. The dissociation is caught deterministically rather than statistically.

For production deployments where standalone signing is not enough &mdash; multi-tenant policy, ingest schema enforcement, signed audit evidence &mdash; the deployable proxy is at [vectorpin.dev](https://vectorpin.dev).

---

<div align="center">

### Learn more

[**vectorsmuggle.org**](https://vectorsmuggle.org) &nbsp;·&nbsp; [**GitHub: jaschadub/VectorSmuggle**](https://github.com/jaschadub/VectorSmuggle) &nbsp;·&nbsp; [**Paper (Zenodo DOI)**](https://doi.org/10.5281/zenodo.20058255) &nbsp;·&nbsp; [**VectorPin (defense)**](https://github.com/ThirdKeyAI/VectorPin)

[![Website](https://img.shields.io/badge/website-vectorsmuggle.org-1f6feb?style=flat-square)](https://vectorsmuggle.org)
[![GitHub](https://img.shields.io/badge/source-GitHub-181717?style=flat-square&logo=github)](https://github.com/jaschadub/VectorSmuggle)
[![Paper](https://img.shields.io/badge/paper-Zenodo-3b82f6?style=flat-square)](https://doi.org/10.5281/zenodo.20058255)

</div>

---

> **⚠️ Ethical use only.** This demonstration is for educational and security-research purposes.